# Test the Gym Environment

Sanity-checks `UniWheelEnv` before handing it to PPO: spaces look right, `check_env`
passes, and a random policy falls over quickly (expected - that's the whole reason
stage 02 exists, see `STAGE01_FINDINGS.md`'s gyroscopic-coupling result).


In [ ]:
# Import standard libraries
from pathlib import Path
import time

# Third-party libraries
from gymnasium.utils.env_checker import check_env
import numpy as np


In [ ]:
# Import the custom environment
from uniwheel_env import UniWheelEnv


In [ ]:
# Settings
MJCF_PATH = Path("../../mechanical/urdf/uniwheel/uniwheel.xml")
SEED = 42


In [ ]:
# Initialize environment
env = UniWheelEnv(mjcf_path=MJCF_PATH, render_mode="human")

# Print observation and action spaces
print(env.observation_space)
print(env.action_space)


In [ ]:
# Use gymnasium's built-in environment checker
check_env(env)


In [ ]:
# Reset environment and print observation
obs, info = env.reset(seed=SEED)
print(obs)


In [ ]:
# Manual step with no action (don't turn the motors)
obs, reward, terminated, truncated, info = env.step(np.array([0.0, 0.0]))
print(f"obs: {obs}")
print(f"reward: {reward}")
print(f"terminated: {terminated}")
print(f"truncated: {truncated}")


In [ ]:
# Test: random policy
obs, _ = env.reset()
total_reward = 0

# Go for 200 steps
terminated = False
truncated = False
for step in range(200):
    step_start = time.time()

    # Choose a random action
    action = env.action_space.sample()

    # Take a step with the given action
    obs, reward, terminated, truncated, _ = env.step(action)

    # Accumulate reward
    total_reward += reward

    # Render in MuJoCo
    env.render()

    # Show that we stopped - a random policy falling in well under 200 steps is expected
    # here (see the module docstring / STAGE01_FINDINGS.md), not a bug
    if terminated or truncated:
        print(f"Episode ended at step {step}")
        break

    # Make sure the loop iteration time matches the MJCF timestep time (5 ms)
    slack = env.model.opt.timestep - (time.time() - step_start)
    if slack > 0:
        time.sleep(slack)

print(f"Final obs: {obs}")
print(f"Final reward: {total_reward}")


In [ ]:
# Close the environment
env.close()
